# Phase 2 — Data Profiling & Quality Inspection Notebook
## SIH PS 26102: AI-Powered MPLADS Monitoring Platform

This notebook performs comprehensive data profiling on the processed MPLADS datasets generated by the Phase 2 ingestion pipeline (`data_pipeline/`).

### Active Scope Configuration:
- **Active Datasets**: `LokSabha18`, `RajyaSabha_Sitting`  
- **Optional / Historical Datasets**: `LokSabha17`, `RajyaSabha_Retired`  

### Objectives:
1. Inspect active vs historical scope configuration and availability status.
2. Inspect raw vs. processed row counts & Grand Total row removal.
3. Verify Work ID regex extraction (`WS/MP...`), tab cleaning, and `NA-` recommendation preservation.
4. Inspect null value distributions and categorical field cardinalities.
5. Analyze financial distributions (Sanction Amounts, Fund Disbursed Amounts).
6. Validate cross-dataset joins (Sanctioned → Expenditure, Completed → Sanctioned, MP Allocation → Sanctioned).
7. Document data-quality realities for Phase 3 feature engineering.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'₹{x:,.2f}' if abs(x) > 1 else f'{x:.4f}')

BASE_DIR = Path('..').resolve() if Path('../data').exists() else Path('.').resolve()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
REPORTS_DIR = BASE_DIR / 'data' / 'reports'

print(f"Base directory: {BASE_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")

## 1. Load Data Quality & Scope Validation Report

In [ ]:
report_file = REPORTS_DIR / 'data_quality_report.json'
with open(report_file, 'r', encoding='utf-8') as f:
    report_data = json.load(f)

print("=== Scope Status ===")
for folder, sinfo in report_data.get('scope_status', {}).items():
    cat = "Active" if sinfo.get('is_active') else "Optional/Historical"
    print(f"[{cat}] {folder}: status='{sinfo.get('status')}' ({sinfo.get('message', '')})")

summary_rows = []
for d in report_data['dataset_validation_reports']:
    wid = d.get('work_id_stats', {})
    summary_rows.append({
        'House': d['house_name'],
        'Dataset': d['dataset_name'],
        'Source File': d['source_file'],
        'Rows Before': d['rows_before'],
        'Grand Total Removed': d['grand_total_removed'],
        'Rows After': d['rows_after'],
        'Valid Work IDs': wid.get('valid_work_ids', 0),
        'NA Recommendations': wid.get('na_recommendations', 0)
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

## 2. Load Processed Active Datasets (LokSabha18 & RajyaSabha_Sitting)

In [ ]:
ls18_dir = PROCESSED_DIR / 'LokSabha18'
rs_dir = PROCESSED_DIR / 'RajyaSabha_Sitting'

df_sanc_ls = pd.read_parquet(ls18_dir / 'works_sanctioned.parquet')
df_exp_ls = pd.read_parquet(ls18_dir / 'expenditures.parquet')

df_sanc_rs = pd.read_parquet(rs_dir / 'works_sanctioned.parquet')
df_exp_rs = pd.read_parquet(rs_dir / 'expenditures.parquet')

print(f"Lok Sabha 18th Sanctioned: {len(df_sanc_ls):,} rows")
print(f"Lok Sabha 18th Expenditures: {len(df_exp_ls):,} rows")
print(f"Rajya Sabha Sitting Sanctioned: {len(df_sanc_rs):,} rows")
print(f"Rajya Sabha Sitting Expenditures: {len(df_exp_rs):,} rows")

## 3. Work ID & NA Recommendation Inspection

In [ ]:
df_recom_ls = pd.read_parquet(ls18_dir / 'works_recommended.parquet')
na_recs = df_recom_ls[df_recom_ls['is_na_recommendation'] == True]
valid_recs = df_recom_ls[df_recom_ls['work_id'].notna()]
print(f"Total LS18 Recommendation Rows: {len(df_recom_ls):,}")
print(f"Valid Work IDs: {len(valid_recs):,} ({len(valid_recs)/len(df_recom_ls)*100:.2f}%)")
print(f"NA Unsanctioned Recommendations: {len(na_recs):,} ({len(na_recs)/len(df_recom_ls)*100:.2f}%)")

## 4. Financial Distribution Summary

In [ ]:
print("=== Sanction Amount Summary (Lok Sabha 18th) ===")
print(df_sanc_ls['sanction_amount'].describe())

print("\n=== Sanction Amount Summary (Rajya Sabha Sitting) ===")
print(df_sanc_rs['sanction_amount'].describe())

## 5. Cross-Dataset Join Validation

In [ ]:
joins = report_data['cross_dataset_join_validation']

print("=== Cross Dataset Join Rates (Lok Sabha 18th) ===")
print(f"Sanctioned -> Expenditure Work ID Match: {joins['sanctioned_vs_expenditure']['expenditure_match_percentage']}%")
print(f"Completed -> Sanctioned Work ID Match: {joins['completed_vs_sanctioned']['completed_match_percentage']}%")
print(f"MP Allocation -> Sanctioned MP Match: {joins['mp_allocation_vs_sanctioned']['mp_match_percentage']}%")

## 6. Key Data-Quality Findings & Scope Summary
1. **Scope Configuration**: `LokSabha18` and `RajyaSabha_Sitting` are configured as active target scopes. `LokSabha17` has been archived to `data/processed/_historical/ LokSabha17`.
2. **Grand Total Removal**: Exactly 1 Grand Total row was successfully stripped from each CSV file processed.
3. **Work ID Normalization**: Normalized Work IDs across Lok Sabha 18th and Rajya Sabha Sitting. 100% of Expenditure Work IDs match to Sanctioned Work IDs.
4. **Preservation of NA Recommendations**: Unsanctioned recommendations were preserved with `work_id = NULL` and `is_na_recommendation = True`.
5. **Ready for Phase 3**: Datasets are exported to `.parquet` in `data/processed/` with standardized schema.